In [ ]:
# !pip install playwright

In [ ]:
# !playwright install

# !playwright install chromium

In [ ]:
from playwright.async_api import async_playwright

p = await async_playwright().start()
browser = await p.chromium.launch(headless=False, slow_mo=500)
page = await browser.new_page()

await page.goto("https://www.whitehouse.gov/presidential-actions/executive-orders/")

await page.get_by_role("link", name="Next", exact=True).click()

print(page.url)

posts = page.locator("li.wp-block-post")

rows = []

for post in await posts.all():
    title = await post.locator("h2.wp-block-post-title a").inner_text()
    link = await post.locator("h2.wp-block-post-title a").get_attribute("href")
    date = await post.locator(".wp-block-post-date").inner_text()
    rows.append({"title": title, "date": date, "link": link})

import pandas as pd
df = pd.DataFrame(rows)
df

In [ ]:
async def scrape_page(page):
    rows = []
    posts = page.locator("li.wp-block-post")

    for post in await posts.all():
        link_element = post.locator("h2.wp-block-post-title a")

        name = await link_element.inner_text()
        link = await link_element.get_attribute("href")
        date = await post.locator(".wp-block-post-date").inner_text()

        rows.append({"name": name, "date": date, "link": link})

    df=pd.DataFrame(rows)

    return df

In [ ]:
p = await async_playwright().start()
browser = await p.chromium.launch(headless=False, slow_mo=500)
page = await browser.new_page()

await page.goto("https://www.whitehouse.gov/presidential-actions/executive-orders/")

first_page = await scrape_page(page)
first_page

In [ ]:
import asyncio
import random

p = await async_playwright().start()
browser = await p.chromium.launch(headless=False, slow_mo=500)
page = await browser.new_page()

await page.goto("https://www.whitehouse.gov/presidential-actions/executive-orders/")

dfs = []
page_num = 1

while True:
    df = await scrape_page(page)
    dfs.append(df)

    # Stop if there's no "Next" link
    next_link = page.get_by_role("link", name="Next", exact=True)
    if await next_link.count() == 0:
        print("Reached the last page.")
        break
        
    # Wait, click "Next", and wait for the new page to load
    await asyncio.sleep(random.uniform(1, 3))
    await next_link.click()
    page_num += 1
    await page.wait_for_url(f"**/page/{page_num}/")


master_df=pd.concat(dfs, ignore_index=True)

master_df.shape